# Flight Delay Risk Platform — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

U.S. DOT/BTS On-Time Reporting Carrier On-Time Performance data, with a temporal 2026 holdout documented in DATA_CARD.md.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'flight_delay_risk'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
"""Command-line entry point for the flight-delay risk project."""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import DataConfig, clean_flights
from src.evaluate import classification_metrics, expected_calibration_error, top_fraction_lift
from src.features import TARGET, add_risk_features
from src.model import threshold_for_capacity
from src.pipeline import PipelineConfig, run_pipeline


def self_test() -> None:
    raw = pd.DataFrame(
        {
            "FlightDate": pd.date_range("2026-01-01", periods=6),
            "Month": [1] * 6,
            "DayOfWeek": [4, 5, 6, 7, 1, 2],
            "Reporting_Airline": ["AA", "AA", "DL", "DL", "UA", "UA"],
            "Origin": ["JFK", "JFK", "ATL", "ATL", "SFO", "SFO"],
            "Dest": ["LAX", "LAX", "ORD", "ORD", "SEA", "SEA"],
            "CRSDepTime": [800, 900, 1000, 1100, 1200, 1300],
            "CRSArrTime": [1100, 1200, 1300, 1400, 1500, 1600],
            "CRSElapsedTime": [360, 360, 180, 180, 120, 120],
            "Distance": [2475, 2475, 606, 606, 679, 679],
            "ArrDelayMinutes": [0, 22, 7, 45, 16, 0],
            "Cancelled": [0] * 6,
            "Diverted": [0] * 6,
        }
    )
    clean = clean_flights(raw)
    featured = add_risk_features(clean)
    assert featured[TARGET].tolist() == [0, 1, 0, 1, 1, 0]

    y = np.asarray([0, 1, 0, 1, 1, 0])
    score = np.asarray([0.10, 0.80, 0.20, 0.90, 0.70, 0.05])
    threshold = threshold_for_capacity(score, 0.50)
    metrics = classification_metrics(y, score, threshold)
    lift = top_fraction_lift(y, score, 0.50)
    ece = expected_calibration_error(y, score)
    assert metrics["pr_auc"] > metrics["prevalence"]
    assert lift["lift"] >= 1.0
    assert 0.0 <= ece <= 1.0
    print("Flight-delay project self-test passed.")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train and verify the 2026 flight-delay risk classifier")
    parser.add_argument("--self-test", action="store_true", help="Run fast offline checks only")
    parser.add_argument("--year", type=int, default=2026)
    parser.add_argument("--alert-capacity", type=float, default=0.20)
    parser.add_argument("--cache-dir", type=Path, default=Path("data/bts_cache"))
    parser.add_argument("--output-dir", type=Path, default=Path("artifacts/flight_delay_risk"))
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.self_test:
        self_test()
        return

    config = PipelineConfig(
        alert_capacity=args.alert_capacity,
        output_dir=args.output_dir,
        data=DataConfig(year=args.year, cache_dir=args.cache_dir),
    )
    result = run_pipeline(config)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


### `api.py`


In [ ]:
"""FastAPI service for the verified flight-delay risk model."""
from __future__ import annotations

from functools import lru_cache
from typing import Annotated

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator

from src.inference import load_release_artifacts, score_records

app = FastAPI(
    title="Flight Delay Risk API",
    version="1.0.0",
    description="Schedule-time risk scoring for the verified 2026 flight-delay portfolio model.",
)


class FlightRequest(BaseModel):
    flight_date: str
    carrier: str = Field(min_length=1, max_length=8)
    origin: str = Field(min_length=3, max_length=4)
    dest: str = Field(min_length=3, max_length=4)
    crs_dep_minutes: Annotated[int, Field(ge=0, le=1439)]
    crs_arr_minutes: Annotated[int, Field(ge=0, le=1439)]
    crs_elapsed_minutes: Annotated[float, Field(gt=0, le=1500)]
    distance_miles: Annotated[float, Field(gt=0, le=12000)]

    @field_validator("carrier", "origin", "dest")
    @classmethod
    def normalize_codes(cls, value: str) -> str:
        return value.strip().upper()


class BatchRequest(BaseModel):
    flights: list[FlightRequest] = Field(min_length=1, max_length=1000)


@lru_cache(maxsize=1)
def _release_state():
    return load_release_artifacts()


def _require_release():
    try:
        return _release_state()
    except Exception as exc:
        raise HTTPException(
            status_code=503,
            detail=f"Verified model release is not available: {exc}",
        ) from exc


@app.get("/health")
def health() -> dict[str, object]:
    try:
        _, metadata = _release_state()
    except Exception as exc:
        return {"status": "degraded", "model_loaded": False, "detail": str(exc)}

    return {
        "status": "ok",
        "model_loaded": True,
        "verification_pass": metadata.get("verification_pass"),
        "data_year": metadata.get("data_year"),
        "task": metadata.get("task"),
    }


@app.post("/predict")
def predict(request: FlightRequest) -> dict[str, object]:
    model, metadata = _require_release()
    record = request.model_dump()
    score = float(score_records(model, [record])[0])
    threshold = metadata.get("validation_threshold")
    return {
        "risk_score": score,
        "review_threshold": threshold,
        "flag_for_review": bool(threshold is not None and score >= float(threshold)),
        "model_task": metadata.get("task"),
        "data_year": metadata.get("data_year"),
    }


@app.post("/predict-batch")
def predict_batch(request: BatchRequest) -> dict[str, object]:
    model, metadata = _require_release()
    records = [flight.model_dump() for flight in request.flights]
    scores = score_records(model, records)
    threshold = metadata.get("validation_threshold")
    items = [
        {
            "risk_score": float(score),
            "flag_for_review": bool(
                threshold is not None and float(score) >= float(threshold)
            ),
        }
        for score in scores
    ]
    return {
        "count": len(items),
        "review_threshold": threshold,
        "predictions": items,
    }


### `src/data.py`


In [ ]:
"""Download, validate and split official BTS on-time performance data."""
from __future__ import annotations

import zipfile
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import requests

BTS_BASE = "https://transtats.bts.gov/PREZIP"
RAW_COLUMNS = {
    "FlightDate",
    "Month",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "ArrDelayMinutes",
    "Cancelled",
    "Diverted",
}
BASE_COLUMNS = [
    "FlightDate",
    "month",
    "day_of_week",
    "carrier",
    "origin",
    "dest",
    "route",
    "crs_dep_minutes",
    "crs_arr_minutes",
    "crs_elapsed_minutes",
    "distance_miles",
    "delay_minutes",
]


@dataclass(frozen=True)
class DataConfig:
    year: int = 2026
    train_months: tuple[int, ...] = (1, 2, 3)
    validation_months: tuple[int, ...] = (4,)
    test_months: tuple[int, ...] = (5,)
    max_rows_per_train_month: int | None = 120_000
    max_rows_validation: int | None = 120_000
    max_rows_test: int | None = 180_000
    seed: int = 42
    cache_dir: Path = Path("data/bts_cache")
    request_timeout_seconds: int = 120

    def validate(self) -> None:
        month_groups = self.train_months + self.validation_months + self.test_months
        if not month_groups:
            raise ValueError("At least one month is required")
        if any(month not in range(1, 13) for month in month_groups):
            raise ValueError("All months must be between 1 and 12")
        if set(self.train_months) & set(self.validation_months):
            raise ValueError("Train and validation months overlap")
        if set(self.train_months) & set(self.test_months):
            raise ValueError("Train and test months overlap")
        if set(self.validation_months) & set(self.test_months):
            raise ValueError("Validation and test months overlap")


def hhmm_to_minutes(series: pd.Series) -> pd.Series:
    """Convert BTS HHMM schedule fields to minutes after midnight."""
    values = pd.to_numeric(series, errors="coerce").fillna(0).astype(int)
    hours = (values // 100).clip(0, 23)
    minutes = (values % 100).clip(0, 59)
    return hours * 60 + minutes


def bts_zip_url(year: int, month: int) -> str:
    if month not in range(1, 13):
        raise ValueError(f"month must be 1..12; got {month}")
    return (
        f"{BTS_BASE}/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_"
        f"{year}_{month}.zip"
    )


def download_bts_month(
    year: int,
    month: int,
    cache_dir: Path,
    timeout: int = 120,
) -> pd.DataFrame:
    """Download one BTS month once, then reuse the cached zip on later runs."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    zip_path = cache_dir / f"bts_on_time_{year}_{month:02d}.zip"

    if not zip_path.exists():
        url = bts_zip_url(year, month)
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        zip_path.write_bytes(response.content)

    with zipfile.ZipFile(zip_path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        if not csv_names:
            raise RuntimeError(f"No CSV found inside {zip_path}")
        with archive.open(csv_names[0]) as handle:
            frame = pd.read_csv(
                handle,
                usecols=lambda name: name.strip() in RAW_COLUMNS,
                low_memory=False,
            )

    frame.columns = [name.strip() for name in frame.columns]
    missing = sorted(RAW_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"BTS {year}-{month:02d} is missing expected fields: {missing}")
    return frame


def clean_flights(frame: pd.DataFrame) -> pd.DataFrame:
    """Clean raw records and keep only fields available at schedule time plus the target."""
    missing = sorted(RAW_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"Input missing required columns: {missing}")

    data = frame.copy()
    data["FlightDate"] = pd.to_datetime(data["FlightDate"], errors="coerce")
    numeric_columns = [
        "Month",
        "DayOfWeek",
        "CRSDepTime",
        "CRSArrTime",
        "CRSElapsedTime",
        "Distance",
        "ArrDelayMinutes",
        "Cancelled",
        "Diverted",
    ]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data = data.loc[
        data["FlightDate"].notna()
        & data["ArrDelayMinutes"].notna()
        & data["Cancelled"].fillna(0).eq(0)
        & data["Diverted"].fillna(0).eq(0)
    ].copy()

    data["month"] = data["Month"].astype("Int64")
    data["day_of_week"] = data["DayOfWeek"].astype("Int64")
    data["carrier"] = data["Reporting_Airline"].fillna("UNKNOWN").astype(str)
    data["origin"] = data["Origin"].fillna("UNKNOWN").astype(str)
    data["dest"] = data["Dest"].fillna("UNKNOWN").astype(str)
    data["route"] = data["origin"] + "→" + data["dest"]
    data["crs_dep_minutes"] = hhmm_to_minutes(data["CRSDepTime"])
    data["crs_arr_minutes"] = hhmm_to_minutes(data["CRSArrTime"])
    data["crs_elapsed_minutes"] = data["CRSElapsedTime"].clip(lower=1)
    data["distance_miles"] = data["Distance"].clip(lower=0)
    data["delay_minutes"] = data["ArrDelayMinutes"].clip(lower=0)

    data = data.dropna(subset=BASE_COLUMNS).reset_index(drop=True)
    if data.empty:
        raise ValueError("No usable completed flights remain after cleaning")
    if not data["delay_minutes"].ge(0).all():
        raise AssertionError("Delay target must be non-negative")
    if not data["distance_miles"].ge(0).all():
        raise AssertionError("Distance must be non-negative")
    if not data["crs_elapsed_minutes"].gt(0).all():
        raise AssertionError("Scheduled elapsed time must be positive")
    return data[BASE_COLUMNS]


def deterministic_sample(frame: pd.DataFrame, n: int | None, seed: int) -> pd.DataFrame:
    if n is None or len(frame) <= n:
        return frame.copy()
    return frame.sample(n=n, random_state=seed).copy()


def _check_temporal_order(train: pd.DataFrame, valid: pd.DataFrame, test: pd.DataFrame) -> None:
    if not train["FlightDate"].max() < valid["FlightDate"].min():
        raise AssertionError("Training dates overlap validation dates")
    if not valid["FlightDate"].max() < test["FlightDate"].min():
        raise AssertionError("Validation dates overlap test dates")


def load_temporal_splits(config: DataConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load and clean all requested months, then enforce a strict temporal holdout."""
    config.validate()
    months = sorted(set(config.train_months + config.validation_months + config.test_months))
    monthly: dict[int, pd.DataFrame] = {}

    for month in months:
        print(f"Loading BTS {config.year}-{month:02d}...")
        raw = download_bts_month(
            config.year,
            month,
            cache_dir=config.cache_dir,
            timeout=config.request_timeout_seconds,
        )
        monthly[month] = clean_flights(raw)
        print(f"  usable rows: {len(monthly[month]):,}")

    train = pd.concat(
        [
            deterministic_sample(monthly[month], config.max_rows_per_train_month, config.seed + month)
            for month in config.train_months
        ],
        ignore_index=True,
    )
    valid = pd.concat([monthly[month] for month in config.validation_months], ignore_index=True)
    test = pd.concat([monthly[month] for month in config.test_months], ignore_index=True)
    valid = deterministic_sample(valid, config.max_rows_validation, config.seed + 100)
    test = deterministic_sample(test, config.max_rows_test, config.seed + 200)

    _check_temporal_order(train, valid, test)
    return train, valid, test


### `src/features.py`


In [ ]:
"""Leakage-safe schedule-time feature engineering."""
from __future__ import annotations

import numpy as np
import pandas as pd

TARGET = "delay15"
BASE_FEATURES = [
    "month",
    "day_of_week",
    "carrier",
    "origin",
    "dest",
    "route",
    "crs_dep_minutes",
    "crs_arr_minutes",
    "crs_elapsed_minutes",
    "distance_miles",
    "dep_sin",
    "dep_cos",
]
EXTRA_NUMERIC = ["day_of_month", "dep_hour", "arr_sin", "arr_cos", "is_weekend"]
EXTRA_CATEGORICAL = ["carrier_route", "route_dep_block"]
FEATURES = BASE_FEATURES + EXTRA_NUMERIC + EXTRA_CATEGORICAL
CATEGORICAL_FEATURES = ["carrier", "origin", "dest", "route"] + EXTRA_CATEGORICAL


def add_risk_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create the binary target and schedule-time features used by the classifier."""
    required = {
        "FlightDate",
        "delay_minutes",
        "month",
        "day_of_week",
        "carrier",
        "origin",
        "dest",
        "route",
        "crs_dep_minutes",
        "crs_arr_minutes",
        "crs_elapsed_minutes",
        "distance_miles",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Feature input is missing columns: {missing}")

    out = frame.copy()
    out[TARGET] = out["delay_minutes"].ge(15).astype(int)
    out["day_of_month"] = out["FlightDate"].dt.day.astype(int)
    out["dep_hour"] = (out["crs_dep_minutes"] // 60).astype(int)

    dep_angle = 2 * np.pi * out["crs_dep_minutes"] / (24 * 60)
    arr_angle = 2 * np.pi * out["crs_arr_minutes"] / (24 * 60)
    out["dep_sin"] = np.sin(dep_angle)
    out["dep_cos"] = np.cos(dep_angle)
    out["arr_sin"] = np.sin(arr_angle)
    out["arr_cos"] = np.cos(arr_angle)
    out["is_weekend"] = out["day_of_week"].isin([6, 7]).astype(int)
    out["carrier_route"] = out["carrier"].astype(str) + "|" + out["route"].astype(str)
    out["route_dep_block"] = out["route"].astype(str) + "|h" + out["dep_hour"].astype(str)

    if not set(out[TARGET].unique()).issubset({0, 1}):
        raise AssertionError("Binary target contains values outside {0, 1}")
    return out


### `src/model.py`


In [ ]:
"""Model configuration and threshold selection."""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from catboost import CatBoostClassifier


@dataclass(frozen=True)
class ModelConfig:
    iterations: int = 900
    learning_rate: float = 0.055
    depth: int = 8
    l2_leaf_reg: float = 10.0
    random_strength: float = 0.4
    early_stopping_rounds: int = 70
    seed: int = 42


def build_model(config: ModelConfig = ModelConfig()) -> CatBoostClassifier:
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=config.iterations,
        learning_rate=config.learning_rate,
        depth=config.depth,
        l2_leaf_reg=config.l2_leaf_reg,
        random_seed=config.seed,
        random_strength=config.random_strength,
        od_type="Iter",
        od_wait=config.early_stopping_rounds,
        verbose=100,
        allow_writing_files=False,
    )


def threshold_for_capacity(scores: np.ndarray, capacity: float) -> float:
    """Choose the validation threshold that flags approximately `capacity` of flights."""
    if not 0.01 <= capacity <= 0.90:
        raise ValueError("capacity must be between 0.01 and 0.90")
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        raise ValueError("scores must not be empty")
    threshold = float(np.quantile(scores, 1.0 - capacity))
    return float(np.clip(threshold, 0.0, 1.0))


### `src/pipeline.py`


In [ ]:
"""End-to-end training, evaluation and evidence generation."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss

from .data import DataConfig, load_temporal_splits
from .evaluate import (
    calibration_table,
    capacity_curve,
    classification_metrics,
    expected_calibration_error,
)
from .features import CATEGORICAL_FEATURES, FEATURES, TARGET, add_risk_features
from .model import ModelConfig, build_model, threshold_for_capacity


@dataclass(frozen=True)
class PipelineConfig:
    alert_capacity: float = 0.20
    output_dir: Path = Path("artifacts/flight_delay_risk")
    data: DataConfig = DataConfig()
    model: ModelConfig = ModelConfig()

    def validate(self) -> None:
        if not 0.01 <= self.alert_capacity <= 0.90:
            raise ValueError("alert_capacity must be between 0.01 and 0.90")
        self.data.validate()


def _constant_baseline(y_train: np.ndarray, y_test: np.ndarray) -> dict[str, float]:
    prevalence = float(y_train.mean())
    scores = np.full(len(y_test), prevalence, dtype=float)
    return {
        "pr_auc": float(average_precision_score(y_test, scores)),
        "roc_auc": 0.5,
        "log_loss": float(log_loss(y_test, scores, labels=[0, 1])),
        "brier": float(brier_score_loss(y_test, scores)),
        "train_prevalence_score": prevalence,
        "test_prevalence": float(y_test.mean()),
    }


def run_pipeline(config: PipelineConfig = PipelineConfig()) -> dict[str, object]:
    config.validate()
    config.output_dir.mkdir(parents=True, exist_ok=True)

    train_raw, valid_raw, test_raw = load_temporal_splits(config.data)
    train = add_risk_features(train_raw)
    valid = add_risk_features(valid_raw)
    test = add_risk_features(test_raw)

    x_train, y_train = train[FEATURES], train[TARGET].to_numpy()
    x_valid, y_valid = valid[FEATURES], valid[TARGET].to_numpy()
    x_test, y_test = test[FEATURES], test[TARGET].to_numpy()

    model = build_model(config.model)
    categorical_indices = [FEATURES.index(name) for name in CATEGORICAL_FEATURES]
    model.fit(
        x_train,
        y_train,
        cat_features=categorical_indices,
        eval_set=(x_valid, y_valid),
        use_best_model=True,
    )

    validation_scores = model.predict_proba(x_valid)[:, 1]
    threshold = threshold_for_capacity(validation_scores, config.alert_capacity)
    test_scores = model.predict_proba(x_test)[:, 1]

    test_metrics = classification_metrics(y_test, test_scores, threshold)
    test_metrics["expected_calibration_error_10bin"] = expected_calibration_error(y_test, test_scores)
    baseline = _constant_baseline(y_train, y_test)
    curve = capacity_curve(y_test, test_scores)
    calibration = calibration_table(y_test, test_scores)

    audit = test[["FlightDate", "carrier", "origin", "dest", "route", "delay_minutes"]].copy()
    audit[TARGET] = y_test
    audit["risk_score"] = test_scores
    audit["alert"] = (test_scores >= threshold).astype(int)
    carrier_slices = (
        audit.groupby("carrier", observed=True)
        .agg(
            rows=(TARGET, "size"),
            prevalence=(TARGET, "mean"),
            mean_score=("risk_score", "mean"),
            alert_rate=("alert", "mean"),
        )
        .query("rows >= 500")
        .sort_values("prevalence", ascending=False)
    )

    model_path = config.output_dir / "flight_delay_catboost.cbm"
    model.save_model(model_path)
    curve.to_csv(config.output_dir / "capacity_curve.csv", index=False)
    calibration.to_csv(config.output_dir / "calibration_table.csv", index=False)
    carrier_slices.to_csv(config.output_dir / "carrier_risk_slices.csv")

    reloaded = CatBoostClassifier()
    reloaded.load_model(model_path)
    reloaded_scores = reloaded.predict_proba(x_test.head(500))[:, 1]
    reload_match = bool(np.allclose(reloaded_scores, test_scores[:500], atol=1e-10))

    verification_pass = bool(
        reload_match
        and test_metrics["pr_auc"] >= test_metrics["prevalence"]
        and 0.0 <= test_metrics["roc_auc"] <= 1.0
        and 0.0 <= threshold <= 1.0
    )
    if not verification_pass:
        raise AssertionError("One or more release checks failed")

    curve_records = [
        {
            "fraction": float(row.fraction),
            "rows": int(row.rows),
            "delay_rate": float(row.delay_rate),
            "population_prevalence": float(row.population_prevalence),
            "lift": float(row.lift),
        }
        for row in curve.itertuples(index=False)
    ]

    metadata: dict[str, object] = {
        "project": "Flight Delay Prediction and Risk Analysis",
        "verification_pass": verification_pass,
        "data_source": "US DOT Bureau of Transportation Statistics — On-Time Reporting Carrier Performance",
        "data_year": config.data.year,
        "task": "predict arrival delay >=15 minutes from schedule-time information",
        "split": {
            "train_months": list(config.data.train_months),
            "validation_months": list(config.data.validation_months),
            "test_months": list(config.data.test_months),
        },
        "rows": {"train": len(train), "validation": len(valid), "test": len(test)},
        "features": FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "validation_threshold": threshold,
        "alert_capacity_target": config.alert_capacity,
        "baseline_test": baseline,
        "classifier_test": test_metrics,
        "capacity_curve": curve_records,
        "model_config": asdict(config.model),
        "release_checks": {
            "saved_model_reload_matches": reload_match,
            "pr_auc_at_least_prevalence": bool(test_metrics["pr_auc"] >= test_metrics["prevalence"]),
            "threshold_in_probability_range": bool(0.0 <= threshold <= 1.0),
        },
        "limitations": [
            "Completed non-diverted flights only; cancellations and diversions need separate models.",
            "No weather, aircraft rotation, crew, congestion or live operational features are used.",
            "May 2026 is temporally held out, but future network regimes can drift.",
            "The alert threshold is selected on April for a fixed review capacity and should be monitored after deployment.",
            "This is operational decision support, not a guarantee that a specific flight will be delayed.",
        ],
    }
    (config.output_dir / "verification.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata


### `src/evaluate.py`


In [ ]:
"""Evaluation helpers for ranking, calibration and operational capacity."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)


def classification_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    predicted = scores >= threshold
    prevalence = float(y_true.mean())
    precision = float(precision_score(y_true, predicted, zero_division=0))
    return {
        "pr_auc": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "log_loss": float(log_loss(y_true, scores, labels=[0, 1])),
        "brier": float(brier_score_loss(y_true, scores)),
        "precision": precision,
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "flag_rate": float(predicted.mean()),
        "prevalence": prevalence,
        "precision_lift_vs_prevalence": float(precision / prevalence) if prevalence else float("nan"),
    }


def top_fraction_lift(y_true: np.ndarray, scores: np.ndarray, fraction: float) -> dict[str, float]:
    if not 0 < fraction <= 1:
        raise ValueError("fraction must be in (0, 1]")
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    n = max(1, int(round(len(y_true) * fraction)))
    order = np.argsort(-scores, kind="mergesort")[:n]
    rate = float(y_true[order].mean())
    prevalence = float(y_true.mean())
    return {
        "fraction": float(fraction),
        "rows": int(n),
        "delay_rate": rate,
        "population_prevalence": prevalence,
        "lift": float(rate / prevalence) if prevalence else float("nan"),
    }


def capacity_curve(y_true: np.ndarray, scores: np.ndarray) -> pd.DataFrame:
    """Show how much delay risk is concentrated in the highest-scored flights."""
    rows = [top_fraction_lift(y_true, scores, fraction) for fraction in (0.05, 0.10, 0.20, 0.30, 0.50)]
    return pd.DataFrame(rows)


def expected_calibration_error(y_true: np.ndarray, scores: np.ndarray, bins: int = 10) -> float:
    """Weighted absolute calibration gap across equal-width probability bins."""
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    bucket = np.clip(np.digitize(scores, edges[1:-1], right=False), 0, bins - 1)
    error = 0.0
    for idx in range(bins):
        mask = bucket == idx
        if not np.any(mask):
            continue
        error += float(mask.mean()) * abs(float(scores[mask].mean()) - float(y_true[mask].mean()))
    return float(error)


def calibration_table(y_true: np.ndarray, scores: np.ndarray, bins: int = 10) -> pd.DataFrame:
    """Create a compact reliability table for inspection in the retained artifacts."""
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    bucket = np.clip(np.digitize(scores, edges[1:-1], right=False), 0, bins - 1)
    rows: list[dict[str, float | int]] = []
    for idx in range(bins):
        mask = bucket == idx
        if not np.any(mask):
            continue
        rows.append(
            {
                "bin": idx + 1,
                "rows": int(mask.sum()),
                "mean_predicted_risk": float(scores[mask].mean()),
                "observed_delay_rate": float(y_true[mask].mean()),
            }
        )
    return pd.DataFrame(rows)


### `src/inference.py`


In [ ]:
"""Inference-time feature construction and release artifact loading."""
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

from .features import FEATURES


def build_inference_features(records: Iterable[Mapping[str, object]]) -> pd.DataFrame:
    """Convert schedule-time request records into the exact training feature order."""
    frame = pd.DataFrame(list(records)).copy()
    required = {
        "flight_date",
        "carrier",
        "origin",
        "dest",
        "crs_dep_minutes",
        "crs_arr_minutes",
        "crs_elapsed_minutes",
        "distance_miles",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Inference request is missing fields: {missing}")
    if frame.empty:
        raise ValueError("At least one flight is required")

    date = pd.to_datetime(frame["flight_date"], errors="raise")
    out = pd.DataFrame(index=frame.index)
    out["month"] = date.dt.month.astype(int)
    out["day_of_week"] = (date.dt.dayofweek + 1).astype(int)
    out["carrier"] = frame["carrier"].astype(str).str.strip().str.upper()
    out["origin"] = frame["origin"].astype(str).str.strip().str.upper()
    out["dest"] = frame["dest"].astype(str).str.strip().str.upper()
    out["route"] = out["origin"] + "-" + out["dest"]

    for name in [
        "crs_dep_minutes",
        "crs_arr_minutes",
        "crs_elapsed_minutes",
        "distance_miles",
    ]:
        out[name] = pd.to_numeric(frame[name], errors="raise")

    out["day_of_month"] = date.dt.day.astype(int)
    out["dep_hour"] = (out["crs_dep_minutes"] // 60).astype(int)

    dep_angle = 2 * np.pi * out["crs_dep_minutes"] / (24 * 60)
    arr_angle = 2 * np.pi * out["crs_arr_minutes"] / (24 * 60)
    out["dep_sin"] = np.sin(dep_angle)
    out["dep_cos"] = np.cos(dep_angle)
    out["arr_sin"] = np.sin(arr_angle)
    out["arr_cos"] = np.cos(arr_angle)
    out["is_weekend"] = out["day_of_week"].isin([6, 7]).astype(int)
    out["carrier_route"] = out["carrier"] + "|" + out["route"]
    out["route_dep_block"] = out["route"] + "|h" + out["dep_hour"].astype(str)

    return out[FEATURES]


def load_release_artifacts(
    model_path: str | Path | None = None,
    metadata_path: str | Path | None = None,
) -> tuple[CatBoostClassifier, dict[str, object]]:
    """Load the trained CatBoost model and its release metadata."""
    model_file = Path(
        model_path
        or os.getenv(
            "FLIGHT_DELAY_MODEL_PATH",
            "artifacts/flight_delay_risk/flight_delay_catboost.cbm",
        )
    )
    metadata_file = Path(
        metadata_path
        or os.getenv(
            "FLIGHT_DELAY_METADATA_PATH",
            "artifacts/flight_delay_risk/verification.json",
        )
    )
    if not model_file.is_file():
        raise FileNotFoundError(f"Model artifact not found: {model_file}")
    if not metadata_file.is_file():
        raise FileNotFoundError(f"Release metadata not found: {metadata_file}")

    model = CatBoostClassifier()
    model.load_model(model_file)
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    if metadata.get("verification_pass") is not True:
        raise ValueError("Release metadata is not marked verification_pass=true")
    return model, metadata


def score_records(
    model: CatBoostClassifier,
    records: Iterable[Mapping[str, object]],
) -> np.ndarray:
    features = build_inference_features(records)
    return np.asarray(model.predict_proba(features)[:, 1], dtype=float)


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


## Application engineering layer

The original project above is intentionally preserved. The cells below expose additional canonical Python from this same project—pipelines, APIs, feature code, evaluation, tests, monitoring and other application logic—so the notebook works as a single recruiter-facing project while the modular files remain the production source of truth.


### Canonical source: `src/__init__.py`


In [ ]:
"""Reusable components for the flight-delay risk project."""


### Canonical source: `tests/test_core.py`


In [ ]:
from __future__ import annotations

import unittest

import numpy as np
import pandas as pd

from src.data import hhmm_to_minutes
from src.evaluate import expected_calibration_error, top_fraction_lift
from src.model import threshold_for_capacity


class CoreTests(unittest.TestCase):
    def test_hhmm_conversion(self) -> None:
        values = pd.Series([0, 5, 930, 2359, None])
        self.assertEqual(hhmm_to_minutes(values).tolist(), [0, 5, 570, 1439, 0])

    def test_capacity_threshold_is_valid_probability(self) -> None:
        score = np.asarray([0.1, 0.2, 0.3, 0.8, 0.9])
        threshold = threshold_for_capacity(score, 0.40)
        self.assertGreaterEqual(threshold, 0.0)
        self.assertLessEqual(threshold, 1.0)

    def test_lift_rewards_good_ranking(self) -> None:
        y = np.asarray([1, 1, 0, 0])
        score = np.asarray([0.9, 0.8, 0.2, 0.1])
        result = top_fraction_lift(y, score, 0.50)
        self.assertEqual(result["delay_rate"], 1.0)
        self.assertEqual(result["lift"], 2.0)

    def test_calibration_error_range(self) -> None:
        y = np.asarray([0, 0, 1, 1])
        score = np.asarray([0.1, 0.2, 0.8, 0.9])
        ece = expected_calibration_error(y, score, bins=4)
        self.assertGreaterEqual(ece, 0.0)
        self.assertLessEqual(ece, 1.0)


if __name__ == "__main__":
    unittest.main()


### Canonical source: `tests/test_inference.py`


In [ ]:
from __future__ import annotations

import unittest

from src.features import FEATURES
from src.inference import build_inference_features


class InferenceFeatureTests(unittest.TestCase):
    def test_request_builds_exact_training_schema(self) -> None:
        frame = build_inference_features(
            [
                {
                    "flight_date": "2026-05-17",
                    "carrier": "aa",
                    "origin": "jfk",
                    "dest": "lax",
                    "crs_dep_minutes": 480,
                    "crs_arr_minutes": 690,
                    "crs_elapsed_minutes": 390,
                    "distance_miles": 2475,
                }
            ]
        )
        self.assertEqual(list(frame.columns), FEATURES)
        self.assertEqual(frame.loc[0, "carrier"], "AA")
        self.assertEqual(frame.loc[0, "origin"], "JFK")
        self.assertEqual(frame.loc[0, "dest"], "LAX")
        self.assertEqual(frame.loc[0, "route"], "JFK-LAX")
        self.assertEqual(frame.loc[0, "carrier_route"], "AA|JFK-LAX")
        self.assertEqual(frame.loc[0, "dep_hour"], 8)

    def test_missing_required_field_fails(self) -> None:
        with self.assertRaises(ValueError):
            build_inference_features(
                [
                    {
                        "flight_date": "2026-05-17",
                        "carrier": "AA",
                        "origin": "JFK",
                        "dest": "LAX",
                        "crs_dep_minutes": 480,
                        "crs_arr_minutes": 690,
                        "crs_elapsed_minutes": 390,
                    }
                ]
            )

    def test_empty_batch_fails(self) -> None:
        with self.assertRaises(ValueError):
            build_inference_features([])


if __name__ == "__main__":
    unittest.main()


## Portfolio depth check

**Meaningful code lines currently visible in this notebook:** 849. A useful portfolio guide is roughly **500 meaningful lines** for a major notebook, with project-specific depth often ranging from about 250 to 1,200 lines. This notebook is within the substantial-project guide.

Line count is not a quality metric by itself. Additional code should only be added when it strengthens the real application: data validation, cleaning, EDA, feature engineering, modelling, tuning, error analysis, explainability, inference, testing, monitoring, APIs, reproducibility or business logic.
